# Cedar Compass Travel: agent-powered business assistant

EECE 503P/798S - Class 3 assignment. This fictional travel agency uses the OpenAI API, explicit context-based LLM routing, genuine function calls, SQLite persistence, and reflection. Online hosting is deferred. Run this notebook from the `business_bot` directory.

## 1. Setup

Use the environment created in the README. Select that environment as the notebook kernel. Copy `.env.example` to `.env` and set your API key locally. Never print or commit keys. An OpenAI API key is required for the live cells; the automated offline tests do not require one.

In [ ]:
from pathlib import Path
import os, sys, json
from uuid import uuid4
PROJECT = Path.cwd()
if not (PROJECT / 'agent.py').exists():
    PROJECT = PROJECT / 'business_bot'
assert (PROJECT / 'agent.py').exists(), 'Open the notebook from the project directory.'
sys.path.insert(0, str(PROJECT))
from dotenv import load_dotenv
load_dotenv(PROJECT / '.env')
print('API key configured:', bool(os.getenv('OPENAI_API_KEY')))

## 2. Fictional business knowledge

Both the summary and the PDF are read into context. There is no document-search tool or vector database. The PDF is built from `me/business_profile.txt`.

In [ ]:
from agent import load_business_context
knowledge = load_business_context(PROJECT / 'me')
print(knowledge)

## 3. Persona

The identity and tone are defined separately from operational rules.

In [ ]:
from prompts import PERSONA
print(PERSONA)

## 4. Instructions

Operational guardrails govern facts, contact collection, tool use, and unsupported requests. Every unanswered business question starts name/email collection for staff review, without requiring a callback request. Answerable questions need no contact information; customers can decline.

In [ ]:
from prompts import INSTRUCTIONS
print(INSTRUCTIONS)

## 5. Explicit router

The LLM receives business knowledge and chat history and returns a validated intent. Python dispatches that intent to one of three handlers. No keyword rules choose the route. A short observable justification is logged; hidden reasoning is not requested.

In [ ]:
from prompts import ROUTING_INSTRUCTIONS
from schemas import RoutingDecision
print(ROUTING_INSTRUCTIONS)
print(json.dumps(RoutingDecision.model_json_schema(), indent=2))

## 6. Tool definitions and persistence

`record_customer_interest(email, name, message)` saves a lead; `record_feedback(question)` saves feedback or an unanswered question. Schemas below define names, parameters, and result contracts. Python executes actual model-returned `function_call` items and sends matching `function_call_output` items back to the model. SQLite records survive local restarts.

In [ ]:
from schemas import TOOL_SCHEMAS
print(json.dumps(TOOL_SCHEMAS, indent=2))

## 7. Construct the agent

Choose the API model through `OPENAI_MODEL`. API calls use `store=False`; short-term history is passed by this application. The local database also persists demonstration traces.

In [ ]:
from backend import OpenAIBackend
from storage import Storage
from agent import BusinessAgent
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in your local .env first.'
db_path = Path(os.getenv('DATABASE_PATH', 'data/business.sqlite3'))
if not db_path.is_absolute():
    db_path = PROJECT / db_path
storage = Storage(db_path)
backend = OpenAIBackend(os.getenv('OPENAI_MODEL', 'gpt-4o-mini'))
agent = BusinessAgent(backend, storage, knowledge)

## 8. Live conversations

This cell makes billable API calls. Every `ask` call prints the route, tool results, original draft, and revised reply. Only the revised reply enters customer-visible chat history.

In [ ]:
history = []
session_id = str(uuid4())
def ask(message):
    result = agent.chat(message, history, session_id)
    history.extend([{'role':'user','content':message}, {'role':'assistant','content':result.reply}])
    print('Mira:', result.reply)
    print(json.dumps(result.trace, indent=2, ensure_ascii=False))
    return result

In [ ]:
ask('What is included in the Istanbul Culture Escape?')

In [ ]:
ask('I would like a quote for two adults visiting Istanbul in May.')

In [ ]:
ask('My name is Sam Haddad.')

In [ ]:
ask('My email is sam@example.com.')

In [ ]:
ask('Feedback: I wish you had more departure-date options.')

### Unanswered question and automatic contact collection

Start a fresh chat to demonstrate missing-contact collection without an explicit callback request. The question is logged first; the lead is saved once name and email are supplied. Staff review is not an automatic email or guaranteed callback.

In [ ]:
history = []
session_id = str(uuid4())
ask('What is the single-room supplement for Jordan?')

In [ ]:
ask('Sam Haddad')

In [ ]:
ask('sam@example.com')

## 9. Reflection bonus

A separate model pass checks the draft against the same business facts, instructions, and successful/failed tool outcomes. It returns a final reply and concise issues. Original and revised replies are persisted. If it discovers an unlogged unknown question, a bounded correction requests the feedback tool and reflects again. Reflection itself cannot execute recording tools.

In [ ]:
from prompts import REFLECTION_INSTRUCTIONS
print(REFLECTION_INSTRUCTIONS)
reflections = [e for e in storage.records('events', 100) if e['event_type'] == 'reflection']
for event in reflections[:3]:
    print(json.dumps(json.loads(event['payload_json']), indent=2))

## 10. Inspect saved records (owner/developer only)

These records are not displayed in the customer interface.

In [ ]:
print('Leads:', json.dumps(storage.records('leads'), indent=2))
print('Feedback:', json.dumps(storage.records('feedback'), indent=2))

## 11. Gradio chatbot

The local UI uses `ChatInterface`, independent session state, and a development-only trace accordion. You can alternatively run `python app.py` in a terminal. No public sharing is enabled.

In [ ]:
from app import build_app
from ui import ui_launch_options
demo = build_app(agent)
demo.launch(server_name='127.0.0.1', share=False, **ui_launch_options())

## 12. Validation and submission

Run `python -m unittest discover -s tests -v` for meaningful offline tests. These validate persistence, tool round trips, session isolation, errors, and reflection with scripted model responses; they do not establish live model quality. Follow `DEMO.md` for live acceptance testing and the submission video. Submit the repo, this notebook, the business summary/PDF, and a video. Do not submit `.env` or customer database files. Online hosting remains deferred.